# Tutorial 0: Camera Calibration with Anipose

**Pipeline Stage:** Producing the `calibration.toml` that 3D triangulation depends on

---

## Why This Tutorial Exists

Tutorials 1–3 take you from raw multi-camera video all the way to 3D skeletons.
But **Tutorial 3 (3D Triangulation) assumes you already have a `calibration.toml`**
for your camera rig — and never shows you how to make one.

This tutorial fills that gap. It takes you from **raw calibration videos** (one per
camera, all recording the same moving calibration board) to a finished
**`calibration.toml`**, using [Anipose](https://github.com/lambdaloop/anipose) /
[`sleap-anipose`](https://github.com/talmolab/sleap-anipose).

```
┌──────────────────────┐     ┌──────────────────────┐     ┌──────────────────────┐
│  Tutorial 0 (HERE)   │ ──► │  Tutorials 1 & 2     │ ──► │  Tutorial 3          │
│  Camera Calibration  │     │  2D Pose + ReID       │     │  3D Triangulation    │
│  → calibration.toml  │     │                       │     │  (uses calibration)  │
└──────────────────────┘     └──────────────────────┘     └──────────────────────┘
```

## What Calibration Actually Solves

To triangulate a 2D point seen in several cameras into a single 3D point, you must
know, for every camera:

1. **Intrinsics** — focal length, principal point, and lens distortion. *How does a
   3D point in front of this camera land on a pixel?*
2. **Extrinsics** — rotation and translation. *Where is this camera sitting in the
   world, and which way is it pointing?*

Calibration recovers all of these at once by watching a **known object** — a
calibration board with a precisely known geometry — move through the shared field of
view of every camera. Anipose detects the board corners in each view and runs
**iterative bundle adjustment** to jointly solve for every camera's intrinsics and
extrinsics while minimizing **reprojection error** (the pixel gap between where a
corner *was* detected and where the solved model says it *should* be).

## What You Will Learn

1. **Installation** — git clone → conda env → `sleap-anipose`, end to end
2. **The calibration board** — what a ChArUco board is, and printing your own
3. **Recording a good calibration video** — the single biggest driver of quality
4. **The session folder format** — camera subfolders + `calibration_images/`
5. **Handing videos to calibration** — placing your whole clips where `calibrate()`
   looks, unmodified, and why frame alignment across cameras is non-negotiable
6. **Running calibration** — one `slap.calibrate(...)` call → `calibration.toml`
7. **Reading `calibration.toml`** — what every field means
8. **Judging quality** — reprojection-error histogram + overlays, and what's "good"

---

## Part 0: Installation (from scratch)

Calibration uses `sleap-anipose`, which wraps the `aniposelib` calibration engine.
Below is a complete setup starting from nothing. You only need to do this **once**.

> If you already made the `multicam-pose` environment from the main `README.md`, you
> can reuse it — just make sure `sleap-anipose` is installed in it (see step 4).

### 1. Install a conda/mamba distribution (skip if you already have one)

If you don't have `conda`, install [Miniforge](https://github.com/conda-forge/miniforge)
(recommended — ships the fast `mamba` solver and the conda-forge channel by default).

### 2. Clone the repositories

```bash
# The tutorials repo (this folder lives inside it) and sleap-anipose for reference
git clone https://github.com/talmolab/sleap-anipose.git
git clone https://github.com/lambdaloop/anipose.git   # optional: docs & reference
```

You do **not** need to install from the clone — `sleap-anipose` is on PyPI — but the
clone gives you `docs/FOLDER_STRUCTURE.md` and the source to read.

### 3. Create and activate the conda environment

```bash
conda create -n sleap-anipose python=3.9 -y
conda activate sleap-anipose
```

### 4. Install the calibration stack

```bash
# Core calibration engine (pulls in aniposelib, opencv, numba, toml, imageio, etc.)
pip install sleap-anipose

# Notebook + plotting utilities used in this tutorial
pip install jupyter ipykernel matplotlib pandas
```

### 5. Register a Jupyter kernel so this notebook can find the env

```bash
python -m ipykernel install --user \
    --name sleap-anipose \
    --display-name "Python (sleap-anipose)"
```

Then, in Jupyter, pick the **"Python (sleap-anipose)"** kernel (top-right) before
running the cells below.

> **OpenCV / ArUco note:** `sleap-anipose` depends on `opencv-contrib` for the ArUco
> module. `pip install sleap-anipose` handles this. If you ever see
> `module 'cv2' has no attribute 'aruco'`, you have a plain `opencv-python` shadowing
> it — fix with:
> `pip uninstall -y opencv-python opencv-contrib-python && pip install opencv-contrib-python`.

In [ ]:
# ============================================================
# STEP 0: Verify the environment
# ============================================================
import sys, platform
print(f"Python:   {sys.version.split()[0]}  ({platform.system()})")

import cv2
print(f"OpenCV:   {cv2.__version__}")
assert hasattr(cv2, "aruco"), (
    "cv2.aruco missing — install opencv-contrib-python (see the note above)."
)

import numpy as np
import matplotlib
print(f"numpy:    {np.__version__}")
print(f"mpl:      {matplotlib.__version__}")

try:
    import sleap_anipose as slap
    import aniposelib
    print(f"sleap-anipose: OK  |  aniposelib: {getattr(aniposelib, '__version__', 'installed')}")
except Exception as e:
    print(f"WARNING: could not import sleap_anipose ({e}).")
    print("Re-check Part 0 and that you selected the 'Python (sleap-anipose)' kernel.")

---

## Part 1: The Calibration Board

Anipose supports **checkerboards**, **ArUco** boards, and **ChArUco** boards.
`sleap-anipose` standardizes on the **ChArUco** board, and so will we.

### Why ChArUco?

A ChArUco board is a chessboard with an ArUco marker inside every white square:

```
┌───┬───┬───┬───┐
│▪ ▪│███│▪ ▪│███│   ███  = black chessboard square
├───┼───┼───┼───┤   ▪ ▪  = ArUco marker (a unique binary tag)
│███│▪ ▪│███│▪ ▪│
├───┼───┼───┼───┤   • Chessboard corners → sub-pixel accurate positions
│▪ ▪│███│▪ ▪│███│   • ArUco tags         → identify WHICH corner is which,
└───┴───┴───┴───┘                          even when the board is partly cut off
```

The chessboard gives **precision**; the ArUco tags give **unique identity** for each
corner. That combination means the board still calibrates correctly even when it's
tilted, partly out of frame, or only partly overlapping between two cameras — which is
exactly what happens when you wave it around a multi-camera volume.

### Board parameters

A ChArUco board is fully described by six numbers. These **must match your physical
board** — most importantly the lengths, which set the real-world **units** of your
entire 3D reconstruction.

| Parameter | Meaning | Example |
|---|---|---|
| `board_x` | squares across the width | `8` |
| `board_y` | squares down the height | `11` |
| `square_length` | chessboard square edge, **in your chosen units** | `24.0` (mm) |
| `marker_length` | ArUco marker edge, same units | `18.75` (mm) |
| `marker_bits` | bits per ArUco marker (4/5/6/7) | `4` |
| `dict_size` | ArUco dictionary size (50/100/250/1000) | `1000` |

> **Units set the world scale.** If `square_length` is in millimetres, every 3D
> coordinate you triangulate later (Tutorial 3) will be in millimetres. Measure your
> printed board's square edge with calipers and put the *real* number here.

In [ ]:
# ============================================================
# STEP 1: Define the calibration board
# ============================================================
# EDIT THESE to match YOUR physical board.
BOARD = {
    "board_x": 8,            # squares across (width)
    "board_y": 11,           # squares down (height)
    "square_length": 24.0,   # square edge length -> sets world units (mm here)
    "marker_length": 18.75,  # ArUco marker edge length (same units)
    "marker_bits": 4,        # 4x4 markers
    "dict_size": 1000,       # DICT_4X4_1000
}

# Persist the board spec next to your data so calibration is reproducible.
# This writes a board.toml that slap.calibrate() can also read directly.
import os
PROJECT_DIR = "calibration_demo"            # working directory for this tutorial
os.makedirs(PROJECT_DIR, exist_ok=True)
board_toml = os.path.join(PROJECT_DIR, "board.toml")

slap.write_board(board_name=board_toml, **BOARD)
print(f"Wrote board spec -> {board_toml}")
print(open(board_toml).read())

### Print your own board

You need a **physical** copy of the board to record calibration video. `sleap-anipose`
can draw a printable one for you. Print it at 100% scale (no "fit to page"), mount it
on something **rigid and flat** (foam board, clipboard), then **re-measure** the actual
printed square size and update `square_length` / `marker_length` above if it drifted.

In [ ]:
# ============================================================
# STEP 2: Draw a printable ChArUco board
# ============================================================
board_png = os.path.join(PROJECT_DIR, "charuco_board.png")

slap.draw_board(
    board_name=board_png,
    board_x=BOARD["board_x"],
    board_y=BOARD["board_y"],
    square_length=BOARD["square_length"],
    marker_length=BOARD["marker_length"],
    marker_bits=BOARD["marker_bits"],
    dict_size=BOARD["dict_size"],
    img_width=1440,
    img_height=1980,      # ~ width * board_y / board_x keeps squares square
    save="",              # (optional) path to also dump a board.toml
)

import matplotlib.pyplot as plt
img = plt.imread(board_png)
plt.figure(figsize=(6, 8))
plt.imshow(img, cmap="gray")
plt.title("Printable ChArUco board\n(print at 100% scale, mount flat & rigid)")
plt.axis("off")
plt.show()
print(f"Saved printable board -> {board_png}")

---

## Part 2: Recording a Good Calibration Video

**This is the step that decides your calibration quality.** The math is only as good as
the board coverage you feed it. Record one synchronized clip per camera of the board
being moved slowly through the capture volume.

### The rules that matter

- **Every camera must see the board a lot.** Bundle adjustment can only relate two
  cameras through frames where *both* see the board. Move so that overlapping pairs get
  many shared views.
- **Fill the whole volume.** Walk the board through the entire 3D space your subjects
  will occupy — near/far, left/right, floor/height. Corners of the volume matter most.
- **Tilt and rotate the board.** Vary its angle (pitch/yaw/roll), not just its
  position. Head-on-only views make focal length and distortion poorly constrained.
- **Move slowly / pause.** Rolling-shutter motion blur ruins corner detection. Glide,
  and hold briefly at each pose. Good lighting, no glare on the board.
- **Keep it flat and rigid.** Any bend in the board violates the known geometry and
  poisons the solve.
- **Enough frames.** After keeping only frames where the board is clearly visible, aim
  for **~100–300 good frames per camera**. A 1–3 minute clip usually gets you there.

### Synchronization

The cameras don't need microsecond sync for calibration (the board is roughly static
during each slow pose), but roughly aligned clips help. If your rig hardware-syncs for
the real recordings, just reuse that.

```
        Move the board through the WHOLE volume, tilting as you go:

           near ─────────────────────────► far
            ┌───────────────────────────────┐
            │   ◹      ◺       ◹      ◺      │   each ◹/◺ = board at a
        top │        ◺      every height &  │        different pose
            │   ◺       depth, tilted        │        (position + angle)
     bottom │      ◹        ◺        ◹       │
            └───────────────────────────────┘
```

---

## Part 3: The Session Folder Format

`sleap-anipose` is organized around a **session** folder. Each camera/view gets a
subfolder containing a `calibration_images/` subfolder. `calibrate()` writes
`calibration.toml` at the **session root**.

### Target structure

```
session/                              <- the "session" you pass to calibrate()
├── calibration.toml                  <- OUTPUT (what this tutorial produces)
├── calibration_metadata.h5           <- OUTPUT (detections + reprojections)
├── reprojection_histogram.png        <- OUTPUT (quality plot)
├── CAM1/
│   └── calibration_images/
│       └── session-CAM1-calibration.mp4    <- your calibration video, unmodified
├── CAM2/
│   └── calibration_images/
│       └── session-CAM2-calibration.mp4
├── ...
└── CAM6/
    └── calibration_images/
        └── session-CAM6-calibration.mp4
```

### How `calibrate()` finds the video

Straight from the installed `sleap_anipose/calibration.py`:

```python
calib_video = list(cam.glob("*/*calibration.mp4"))
if not calib_video:
    calib_videos.append([make_calibration_videos(cam.as_posix())])   # stitches *.jpg
else:
    calib_videos.append([calib_video[0].as_posix()])                 # uses yours
```

For each camera folder it looks **one directory deep for a file whose name ends in
`calibration.mp4`**. If it finds one, it uses it as-is. If not, it falls back to
stitching `calibration_images/*.jpg` into a video.

So Part 4's whole job is: **put your video there, under a name ending in
`calibration.mp4`.** We place the original file with a hard link — nothing is
re-encoded, nothing is extracted, no frames are dropped. `aniposelib` reads the whole
clip.

> Two corrections to earlier versions of this tutorial. It said `calibrate()` globs
> `*/*.MOV` — it does not, it globs `*/*calibration.mp4`, so a `.MOV` would be silently
> ignored. And it extracted JPEGs and let `calibrate()` rebuild a video from them, which
> is worth avoiding for the two reasons below.

### Why not extract frames to images

**1. The JPEG round-trip resizes your image.** `make_calibration_videos()` calls
`imageio.get_writer(fname, fps=30)` with no `macro_block_size`, so imageio-ffmpeg **pads
the frame up to a multiple of 16**. A 1920x1080 recording comes back as **1920x1088**.
`calibrate()` then reads the camera resolution from that video, so your intrinsics get
solved for an image 8 px taller than the one your pose data came from — biasing the
principal point `cy`. Plus JPEG is a second lossy generation, and ChArUco corner
detection is sub-pixel sensitive.

**2. Per-camera frame selection silently destroys the extrinsics.** This one matters
most. Bundle adjustment relates two cameras through frames where **both** saw the board,
and in `aniposelib` that pairing is done purely by **frame number**:

```python
# aniposelib/boards.py -- merge_rows()
for cname, rows in zip(cam_names, all_rows):
    for r in rows:
        num = r['framenum']        # (video_index, frame_index)
        rows_dict[cname][num] = r  # grouped across cameras by this key alone
```

No timestamps, no content matching. **Frame `k` of CAM1's video is assumed to be the same
instant as frame `k` of CAM2's.** So if each camera keeps only *its own* good frames and
those get renumbered `0..N`, CAM1 frame 7 and CAM2 frame 7 become unrelated moments, and
the solver is fed board poses that never co-occurred. Per-camera intrinsics survive;
the **extrinsics — the entire point of multi-camera calibration — are quietly wrong.**

Feeding the untouched videos makes this impossible to get wrong: frame `k` already *is*
the same instant in every camera, because that is how they were recorded.

### You do not need to pre-filter boardless frames

`aniposelib` already skips cheaply past them. From `boards.py`:

```python
def detect_video(self, vidname, prefix=None, skip=20, progress=False):
    go = int(skip / 2)
    for framenum in it:
        ...
        if framenum % skip != 0 and go <= 0:
            continue                     # cheap skip when nothing is happening
        corners, ids = self.detect_image(frame)
        if corners is not None and len(corners) > 0:
            go = int(skip / 2)           # board found -> examine the next ~10 densely
```

It samples every 20th frame, and each time it finds the board it densely examines the
following ~10. Frames with no board cost one cheap detection attempt. Pre-filtering buys
nothing and risks the failure above.

### The one requirement: frame-aligned clips

Because frame index is the only cross-camera link, **your calibration clips must start at
the same instant.** If your rig hardware-syncs, you already have this. If not, align the
videos *before* this tutorial (trim them to a common start) — that is a separate,
upstream job, and this notebook will not paper over it. STEP 4c shows you the same frame
from every camera so you can confirm alignment by eye before spending minutes on the
solve.

### Starting point

One raw calibration video per camera, all recording the same board motion at the same
time. Point `RAW_CALIB_VIDEOS` at them; the dict key becomes the camera folder name and
the camera's `name` inside `calibration.toml`.


In [ ]:
# ============================================================
# STEP 3: Point at your calibration videos (one per camera)
# ============================================================
from pathlib import Path

# ── EDIT: view/camera name -> its calibration video ──────────
# These keys become the folder names and the camera names in calibration.toml.
# The videos are used WHOLE and are never modified.
RAW_CALIB_VIDEOS = {
    "CAM1": "/path/to/calibration/CAM1_calib.mp4",
    "CAM2": "/path/to/calibration/CAM2_calib.mp4",
    "CAM3": "/path/to/calibration/CAM3_calib.mp4",
    "CAM4": "/path/to/calibration/CAM4_calib.mp4",
    "CAM5": "/path/to/calibration/CAM5_calib.mp4",
    "CAM6": "/path/to/calibration/CAM6_calib.mp4",
}

# ── Board coverage check (read-only diagnostic, one decode pass) ──
# Tells you whether there is enough board overlap BEFORE you run the solve.
# Set to False to skip it; it does not affect what calibrate() sees.
RUN_BOARD_SCAN = True
SCAN_STRIDE = 15        # examine every Nth frame during the check
MIN_MARKER_FRAC = 0.25  # a view "sees" the board if >= this fraction of markers detected
MIN_CAMS = 2            # a frame is usable if this many views see the board.
                        # 2 is the minimum for triangulation; 3+ gives a stronger solve.

# The session folder we will build and then calibrate.
SESSION = Path(PROJECT_DIR) / "session"
SESSION.mkdir(parents=True, exist_ok=True)

print(f"Session folder: {SESSION.resolve()}\n")
print(f"{'view':6s}  {'status':8s} {'frames':>8} {'resolution':>12} {'fps':>6}  video")
missing = []
VIDEO_INFO = {}
for view, vid in RAW_CALIB_VIDEOS.items():
    p = Path(vid)
    if not p.exists():
        missing.append(view)
        print(f"{view:6s}  {'MISSING':8s} {'-':>8} {'-':>12} {'-':>6}  {vid}")
        continue
    cap = cv2.VideoCapture(str(p))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    VIDEO_INFO[view] = {"n_frames": n, "size": (w, h), "fps": fps}
    print(f"{view:6s}  {'OK':8s} {n:>8} {f'{w}x{h}':>12} {fps:>6.1f}  {p.name}")

if missing:
    print(f"\n(!) Update the paths above for: {', '.join(missing)}")
elif VIDEO_INFO:
    counts = {v: i["n_frames"] for v, i in VIDEO_INFO.items()}
    spread = max(counts.values()) - min(counts.values())
    print(f"\nFrame counts differ by {spread} frame(s) across cameras.")
    if spread > 5:
        print("    Fine if the clips START together and merely stop at different times.")
        print("    A PROBLEM if they start at different instants -- frame index is the")
        print("    only cross-camera link (Part 3), so misaligned clips give quietly")
        print("    wrong extrinsics. Align the clips upstream, then re-run.")
    non_mp4 = [v for v in VIDEO_INFO
               if Path(RAW_CALIB_VIDEOS[v]).suffix.lower() != ".mp4"]
    if non_mp4:
        print(f"\nNote: {non_mp4} are not .mp4. calibrate() only globs for")
        print("    '*calibration.mp4', so STEP 4b places them under an .mp4 name and")
        print("    verifies they still decode (ffmpeg sniffs content, so they usually do).")


---

## Part 4: Put the Videos Where `calibrate()` Looks

That is the entire step:

```
your_videos/CAM1_calib.mp4  ──hard link──►  session/CAM1/calibration_images/session-CAM1-calibration.mp4
your_videos/CAM2_calib.mp4  ──hard link──►  session/CAM2/calibration_images/session-CAM2-calibration.mp4
```

A hard link costs no disk space and copies no bytes (falling back to a symlink, then a
real copy, if the filesystem refuses). Your originals are untouched, and `aniposelib`
reads the **whole clip** — every frame, at the original resolution, with no re-encoding.

### What runs on what

| | |
|---|---|
| Frames `aniposelib` examines | the whole video, at its `skip=20` stride plus dense streaks after each detection |
| Frames we drop | **none** |
| Re-encoding | **none** |
| Images written | **none** |

The optional board-coverage scan in STEP 4b decodes each video once to *report* how much
of the clip actually shows the board. It is a read-only diagnostic — it changes nothing
about what `calibrate()` sees. Its value is telling you "CAM5 only sees the board in 4% of
frames" **before** you wait out a six-camera bundle adjustment.

### Reading the scan output

- **frames by number of cameras seeing the board** — the rows for `MIN_CAMS` and above are
  the frames that can actually constrain the extrinsics. Rows below it contribute to that
  camera's intrinsics only.
- **usable frames** — if this is tiny, bundle adjustment has almost nothing to relate your
  cameras through. Re-record with more shared board coverage; no amount of tuning fixes
  missing data.


In [ ]:
# ============================================================
# STEP 4a: ChArUco detector (handles old & new cv2.aruco APIs)
# ============================================================
import cv2
from cv2 import aruco

_ARUCO_DICTS = {
    (4, 50): aruco.DICT_4X4_50,   (4, 100): aruco.DICT_4X4_100,
    (4, 250): aruco.DICT_4X4_250, (4, 1000): aruco.DICT_4X4_1000,
    (5, 50): aruco.DICT_5X5_50,   (5, 100): aruco.DICT_5X5_100,
    (5, 250): aruco.DICT_5X5_250, (5, 1000): aruco.DICT_5X5_1000,
    (6, 50): aruco.DICT_6X6_50,   (6, 100): aruco.DICT_6X6_100,
    (6, 250): aruco.DICT_6X6_250, (6, 1000): aruco.DICT_6X6_1000,
    (7, 50): aruco.DICT_7X7_50,   (7, 100): aruco.DICT_7X7_100,
    (7, 250): aruco.DICT_7X7_250, (7, 1000): aruco.DICT_7X7_1000,
}

def make_detector(board=BOARD):
    'Return (detect_fn, n_total_markers). detect_fn(gray) -> n_markers_seen.'
    dict_id = _ARUCO_DICTS[(board["marker_bits"], board["dict_size"])]
    aruco_dict = aruco.getPredefinedDictionary(dict_id)

    # New API (OpenCV >= 4.7): ArucoDetector object
    if hasattr(aruco, "ArucoDetector"):
        params = aruco.DetectorParameters()
        det = aruco.ArucoDetector(aruco_dict, params)
        def detect(gray):
            corners, ids, _ = det.detectMarkers(gray)
            return 0 if ids is None else len(ids)
    # Old API (OpenCV <= 4.6): free functions
    else:
        params = aruco.DetectorParameters_create()
        def detect(gray):
            corners, ids, _ = aruco.detectMarkers(gray, aruco_dict, parameters=params)
            return 0 if ids is None else len(ids)

    # A ChArUco board of (bx, by) has floor(bx*by/2) markers.
    n_markers = (board["board_x"] * board["board_y"]) // 2
    return detect, n_markers

_detect_markers, N_MARKERS = make_detector()
print(f"Detector ready. Full board shows up to {N_MARKERS} ArUco markers.")

In [ ]:
# ============================================================
# STEP 4b: Place each whole video where calibrate() will find it
# ============================================================
import os
import shutil

min_markers = max(4, int(MIN_MARKER_FRAC * N_MARKERS))
views = [v for v in RAW_CALIB_VIDEOS if v in VIDEO_INFO]


def place_video(view):
    """Hard-link (or symlink, or copy) the original video into the session.

    Returns (dest_path, how, n_frames). Raises if the placed file will not decode.
    """
    src = Path(RAW_CALIB_VIDEOS[view])
    out_dir = SESSION / view / "calibration_images"
    if out_dir.exists():
        shutil.rmtree(out_dir)               # keep re-runs reproducible
    out_dir.mkdir(parents=True)

    # Name MUST end in "calibration.mp4" for calibrate()'s glob to find it.
    dst = out_dir / f"{SESSION.name}-{view}-calibration.mp4"

    how = None
    for label, fn in (("hard link", os.link),
                      ("symlink", os.symlink),
                      ("copy", shutil.copy2)):
        try:
            fn(str(src), str(dst))
            how = label
            break
        except (OSError, NotImplementedError, AttributeError):
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            continue
    if how is None:
        raise RuntimeError(f"{view}: could not place {src} at {dst}")

    # Verify rather than assume: a silently undecodable video means zero detections
    # and a confusing failure much later.
    cap = cv2.VideoCapture(str(dst))
    opened = cap.isOpened()
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if opened else 0
    got = cap.read()[0] if opened else False
    cap.release()
    if not (opened and got and n > 0):
        raise RuntimeError(
            f"{view}: placed the video but it will not decode "
            f"(opened={opened}, frames={n}). If the source is not really an mp4, "
            f"transcode it to mp4 first and point RAW_CALIB_VIDEOS at that."
        )
    return dst, how, n


print("Placing calibration videos (whole clips, no re-encoding):\n")
CALIB_VIDEOS = {}
for view in views:
    dst, how, n = place_video(view)
    CALIB_VIDEOS[view] = dst
    print(f"  {view:6s} {how:10s} {n:6d} frames -> {dst.relative_to(SESSION)}")
print("\nOriginals untouched. aniposelib will read every one of these frames.")


# ── Optional read-only diagnostic: how much board coverage is there? ──
def scan_camera(view):
    """{frame_index: n_markers} for the frames we examined. Changes nothing."""
    cap = cv2.VideoCapture(str(RAW_CALIB_VIDEOS[view]))
    seen, idx = {}, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % SCAN_STRIDE == 0:
            seen[idx] = _detect_markers(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
        idx += 1
    cap.release()
    return seen


if RUN_BOARD_SCAN:
    print(f"\nBoard coverage check (every {SCAN_STRIDE}th frame; a view 'sees' the "
          f"board with >= {min_markers} of {N_MARKERS} markers):\n")
    scans = {}
    for view in views:
        scans[view] = scan_camera(view)
        n_good = sum(1 for m in scans[view].values() if m >= min_markers)
        pct = n_good / max(len(scans[view]), 1)
        bar = "#" * int(40 * pct)
        print(f"  {view:6s} {n_good:5d}/{len(scans[view]):5d} examined "
              f"({pct:5.1%})  {bar}")

    all_frames = sorted(set().union(*(set(s) for s in scans.values())))
    votes = {f: sum(1 for v in views if scans[v].get(f, 0) >= min_markers)
             for f in all_frames}
    hist = {}
    for f in all_frames:
        hist[votes[f]] = hist.get(votes[f], 0) + 1

    print(f"\n  examined frames by number of cameras seeing the board:")
    for k in sorted(hist, reverse=True):
        tag = "" if k >= MIN_CAMS else "   <- cannot constrain extrinsics"
        print(f"    {k} camera{'s' if k != 1 else ' '}: {hist[k]:5d}{tag}")

    usable = sum(n for k, n in hist.items() if k >= MIN_CAMS)
    print(f"\n  {usable} of {len(all_frames)} examined frames are usable "
          f"(>= {MIN_CAMS} cameras)  ~= {usable * SCAN_STRIDE:,} frames over the "
          f"full clip")
    weak = [v for v in views
            if sum(1 for m in scans[v].values() if m >= min_markers)
            < 0.05 * max(len(scans[v]), 1)]
    if weak:
        print(f"  (!) {weak} see the board in under 5% of frames. Those views will be "
              f"poorly constrained.")
    if usable < 20:
        print("  (!) Very little cross-camera overlap. Bundle adjustment relates cameras")
        print("      only through frames where BOTH see the board -- consider "
              "re-recording.")
else:
    print("\nBoard coverage check skipped (RUN_BOARD_SCAN = False).")


In [ ]:
# ============================================================
# STEP 4c: Verify before spending minutes on the solve
# ============================================================
# Each of these fails silently if you don't check it:
#   1. calibrate()'s glob finds exactly one video per camera
#   2. the placed video decodes
#   3. resolution matches the source (proof nothing re-encoded or padded it)
#   4. the whole clip is there (frame count matches the source)

print(f"{'view':6s} {'glob':>5} {'frames':>8} {'source':>8} {'resolution':>12} "
      f"{'intact':>7}")
problems, notes = [], []
frame_counts = {}

for view in views:
    globbed = list((SESSION / view).glob("*/*calibration.mp4"))   # calibrate()'s glob

    cap = cv2.VideoCapture(str(CALIB_VIDEOS[view]))
    opened = cap.isOpened()
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if opened else 0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) if opened else 0
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) if opened else 0
    cap.release()
    frame_counts[view] = n

    src_n = VIDEO_INFO[view]["n_frames"]
    src_w, src_h = VIDEO_INFO[view]["size"]
    intact = (w, h) == (src_w, src_h) and n == src_n
    print(f"{view:6s} {len(globbed):>5} {n:>8} {src_n:>8} {f'{w}x{h}':>12} "
          f"{'yes' if intact else 'NO':>7}")

    if len(globbed) != 1:
        problems.append(f"{view}: glob found {len(globbed)} videos, expected 1")
    if not opened or n == 0:
        problems.append(f"{view}: video does not decode")
    if (w, h) != (src_w, src_h):
        problems.append(f"{view}: resolution changed {src_w}x{src_h} -> {w}x{h}")
    if n != src_n:
        problems.append(f"{view}: {n} frames but source has {src_n} -- not the whole clip")
    if list((SESSION / view / "calibration_images").glob("*.jpg")):
        problems.append(f"{view}: stray .jpg in calibration_images/ -- calibrate() may "
                        f"rebuild the video from those instead")

spread = max(frame_counts.values()) - min(frame_counts.values())
if spread > 5:
    notes.append(f"frame counts differ by {spread} across cameras: {frame_counts}. "
                 f"Fine if the clips start together; a real problem if they start at "
                 f"different instants (see the grid below).")

print()
if problems:
    print("PROBLEMS:")
    for p in problems:
        print(f"  ! {p}")
else:
    print(f"All {len(views)} videos OK: one per camera, decoding, whole clip, "
          f"original resolution.")
if notes:
    print("\nWORTH CHECKING:")
    for nt in notes:
        print(f"  - {nt}")

# ── The alignment check that actually matters ────────────────
# Same frame index in every camera. The board must be in the SAME real-world pose,
# seen from different angles. This is the only way to catch a sync error by eye.
probe = min(frame_counts.values()) // 2

dict_id = _ARUCO_DICTS[(BOARD["marker_bits"], BOARD["dict_size"])]
aruco_dict = aruco.getPredefinedDictionary(dict_id)
if hasattr(aruco, "ArucoDetector"):
    _d = aruco.ArucoDetector(aruco_dict, aruco.DetectorParameters())
    _detect_full = lambda g: _d.detectMarkers(g)
else:
    _detect_full = lambda g: aruco.detectMarkers(
        g, aruco_dict, parameters=aruco.DetectorParameters_create())

ncol = min(3, len(views))
nrow = int(np.ceil(len(views) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 4.5 * nrow))
axes = np.atleast_1d(axes).ravel()

for ax, view in zip(axes, views):
    cap = cv2.VideoCapture(str(CALIB_VIDEOS[view]))
    cap.set(cv2.CAP_PROP_POS_FRAMES, probe)
    ok, img = cap.read()
    cap.release()
    if not ok:
        ax.text(0.5, 0.5, f"{view}: could not read frame", ha="center", va="center")
        ax.axis("off")
        continue
    corners, ids, _ = _detect_full(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))
    aruco.drawDetectedMarkers(img, corners, ids)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{view}: {0 if ids is None else len(ids)} markers", fontsize=11)
    ax.axis("off")
for ax in axes[len(views):]:
    ax.axis("off")

fig.suptitle(f"Frame {probe} in every camera -- the board should be in the SAME pose, "
             f"seen from different angles", fontsize=12)
plt.tight_layout()
plt.show()
print("If the board sits in visibly different positions across these panels, your clips "
      "are not frame-aligned. Trim them to a common start upstream and re-run -- frame "
      "index is the only cross-camera link, so this cannot be fixed later.")


---

## Part 5: Run Calibration → `calibration.toml`

Everything is now in the format `slap.calibrate()` expects. A single call:

1. Discovers the camera folders (each subfolder of the session is one view).
2. Finds the `*calibration.mp4` we placed in Part 4 — your original video — and reads
   it whole. No JPEG stitching, no resizing, no frames dropped.
3. Detects ChArUco corners across all views and runs **iterative bundle adjustment**,
   pairing cameras by frame index.
4. Writes `calibration.toml` (the deliverable) plus optional QC outputs.

### Key arguments

| Argument | What it does |
|---|---|
| `session` | Path to the session folder (its subfolders are the views) |
| `board` | Board spec — a dict, a `CharucoBoard`, or the `board.toml` path |
| `calib_fname` | Where to write `calibration.toml` (**the main output**) |
| `metadata_fname` | `.h5` of detections + triangulations + reprojections (QC) |
| `histogram_path` | `.png` reprojection-error histogram (QC) |
| `reproj_path` | Folder to write detected-vs-reprojected overlay images (QC) |
| `excluded_views` | View **names** to leave out (e.g. a broken camera) |

> Calibration is CPU-heavy and can take a few minutes for 6 cameras x hundreds of
> frames. That's normal.
>
> `aniposelib` examines every 20th frame by default, but after each successful board
> detection it densely examines the next ~10. Since Part 4 kept only frames where the
> board was actually visible, that streak behaviour means effectively all of them get
> used.

In [ ]:
# ============================================================
# STEP 5: Calibrate the session
# ============================================================
calib_toml   = SESSION / "calibration.toml"
metadata_h5  = SESSION / "calibration_metadata.h5"
histogram_png = SESSION / "reprojection_histogram.png"

cgroup, metadata = slap.calibrate(
    session=str(SESSION),
    board=BOARD,                       # dict is fine; could also pass board_toml
    excluded_views=(),                 # e.g. ("CAM5",) to drop a bad camera
    calib_fname=str(calib_toml),       # <-- produces calibration.toml
    metadata_fname=str(metadata_h5),
    histogram_path=str(histogram_png),
    reproj_path=str(SESSION),          # writes reprojection-*.png into each view
)

frames, detections, triangulations, reprojections = metadata
print(f"\nDone. Calibrated {len(cgroup.get_names())} cameras: {cgroup.get_names()}")
print(f"Common board frames used across all views: {len(frames)}")
print(f"calibration.toml -> {calib_toml.resolve()}")

---

## Part 6: Reading `calibration.toml`

The output is a plain TOML file with one `[cam_...]` block per camera. This is exactly
the file **Tutorial 3** loads to triangulate.

```toml
[cam_0]
name = "CAM1"
size = [1920, 1080]                                 # image resolution (px)
matrix = [[fx, 0, cx], [0, fy, cy], [0, 0, 1]]      # intrinsics
distortions = [k1, k2, p1, p2, k3]                  # lens distortion
rotation = [rx, ry, rz]                             # extrinsics (Rodrigues rvec)
translation = [tx, ty, tz]                          # extrinsics (in board units)
```

- **`matrix`** — intrinsic matrix. `fx, fy` focal lengths (px); `cx, cy` principal point.
- **`distortions`** — radial (`k1,k2,k3`) + tangential (`p1,p2`) lens distortion.
- **`rotation` / `translation`** — where the camera sits in the shared world frame.
  `translation` is in **your board units** (mm if you used mm), which is why the board
  measurement sets your 3D scale.

In [ ]:
# ============================================================
# STEP 6: Print and parse calibration.toml
# ============================================================
import toml

print(calib_toml.read_text()[:2000])
print("..." if calib_toml.stat().st_size > 2000 else "")

calib = toml.load(calib_toml)
print("\nParsed per-camera summary:")
for key, cam in calib.items():
    if key == "metadata":
        continue
    mat = np.array(cam["matrix"])
    print(f"  {cam.get('name', key):6s}  size={cam['size']}  "
          f"fx={mat[0,0]:7.1f}  fy={mat[1,1]:7.1f}  "
          f"cx={mat[0,2]:6.1f}  cy={mat[1,2]:6.1f}  "
          f"|t|={np.linalg.norm(cam['translation']):.1f}")

---

## Part 7: Judging Calibration Quality

**Never trust a calibration you haven't checked.** The single best metric is
**reprojection error**: triangulate the detected board corners back to 3D, project them
into every camera, and measure the pixel distance from the original detections.

### Rules of thumb

| Mean reprojection error | Verdict |
|---|---|
| **< 1 px** | Excellent |
| **1–3 px** | Good — fine for most 3D pose work |
| **3–5 px** | Marginal — usable but consider re-recording |
| **> 5 px** | Poor — re-record with better board coverage, or exclude a bad view |

If one camera is dragging the error up, re-run with that view in `excluded_views`, or
re-record calibration video for it (usually it never shared enough board views with the
others).

In [ ]:
# ============================================================
# STEP 7a: Reprojection-error histogram + per-camera breakdown
# ============================================================
# detections / reprojections: (n_cams, n_frames, n_corners, 2)
err = np.linalg.norm(detections - reprojections, axis=-1)   # (n_cams, n_frames, n_corners)
per_cam = np.nanmean(err.reshape(err.shape[0], -1), axis=1)

print("Per-camera mean reprojection error (px):")
for name, e in zip(cgroup.get_names(), per_cam):
    flag = "  <-- check this view" if e > 5 else ""
    print(f"  {name:6s}  {e:5.2f} px{flag}")
print(f"\nOverall mean: {np.nanmean(err):.2f} px   median: {np.nanmedian(err):.2f} px")

plt.figure(figsize=(8, 5))
plt.hist(err.ravel()[~np.isnan(err.ravel())], bins=np.linspace(0, 15, 60), density=True)
plt.axvline(np.nanmean(err), color="r", ls="--", label=f"mean {np.nanmean(err):.2f}px")
plt.xlabel("Reprojection error (px)")
plt.ylabel("PDF")
plt.title("Reprojection error across all views")
plt.legend()
plt.show()

# The saved histogram from calibrate():
if histogram_png.exists():
    print(f"\nSaved histogram -> {histogram_png}")

In [ ]:
# ============================================================
# STEP 7b: Look at a saved detection-vs-reprojection overlay
# ============================================================
# calibrate(reproj_path=...) drops reprojection-*.png into each view folder.
# Red '+' = detected corners, green 'x' = reprojected corners. They should overlap.
overlays = sorted(SESSION.glob("*/reprojection-*.png"))
if overlays:
    show = overlays[:min(3, len(overlays))]
    fig, axes = plt.subplots(1, len(show), figsize=(6 * len(show), 6))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, show):
        ax.imshow(plt.imread(p))
        ax.set_title(f"{p.parent.name}/{p.name}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No reprojection overlays found (pass reproj_path= to calibrate to generate them).")

---

## Summary & Handoff to Tutorial 3

You went from **raw calibration videos → `calibration.toml`**:

```
per-camera calibration videos  (frame-aligned)
        |  Part 4: hard-link them where calibrate() looks
        v
session/<CAM>/calibration_images/<session>-<CAM>-calibration.mp4
        |         your original file -- whole clip, no re-encode, no extraction
        |  Part 5: slap.calibrate(...)
        v
session/calibration.toml                     <- the deliverable
   + calibration_metadata.h5, reprojection_histogram.png   (QC)
```

### Using it in the pipeline

Drop `calibration.toml` at the root of your **triangulation** session (the folder with
`cam1/ … cam6/` pose `.analysis.h5` files in Tutorial 3) and point the triangulation
step at it:

```python
import sleap_anipose as slap
slap.triangulate(
    p2d="/path/to/triangulation_session",
    calib="/path/to/triangulation_session/calibration.toml",   # <-- from THIS tutorial
    fname="points3d.h5",
)
```

Because `translation` in `calibration.toml` is in the **units of your board's
`square_length`**, your 3D coordinates in Tutorial 3 come out in those same units.

### Checklist for a good calibration

- [ ] Board printed at 100% scale, flat & rigid; `square_length`/`marker_length` measured
- [ ] Calibration clips were recorded **simultaneously** and start at the same instant
- [ ] STEP 4c reports **whole clip, original resolution** for every camera
- [ ] STEP 4c's grid shows the board in the **same pose** in every camera
- [ ] STEP 4b's scan shows plenty of frames seen by **3+** cameras
- [ ] Board was moved through the **whole volume**, tilted at many angles
- [ ] Mean reprojection error **< 3 px**; no single camera is an outlier

> If your reprojection error sits stubbornly in the 5-20 px range, suspect frame
> alignment before you suspect the board or the optics. Per-camera frame selection
> (renumbering each camera's good frames to `0..N` independently) produces exactly that
> signature: plausible intrinsics, quietly wrong extrinsics.
- [ ] `calibration.toml` copied to the triangulation session for Tutorial 3

### Troubleshooting

| Symptom | Likely fix |
|---|---|
| `cv2.aruco` missing | `pip install opencv-contrib-python` (remove plain `opencv-python`) |
| Few/no frames kept | Lower `MIN_MARKER_FRAC` / `FRAME_STRIDE`; check lighting & focus |
| High error on one camera | Add it to `excluded_views`, or re-record that view |
| High error everywhere | Board bent, wrong `square_length`, or too little volume coverage |
| Wrong 3D scale later | `square_length` didn't match the real printed board |